In [2]:
from pathlib import Path
import pandas as pd

# Pandas output formatting
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)

# Self-healing path resolution: find project root containing 'data/processed/'
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed" / "master_cleaned.csv").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

MASTER_CSV = PROJECT_ROOT / "data" / "processed" / "master_cleaned.csv"
QUALITY_CSV = PROJECT_ROOT / "data" / "processed" / "master_quality.csv"

# Verify path before loading
if not MASTER_CSV.exists():
    raise FileNotFoundError(f"Could not locate master_cleaned.csv at: {MASTER_CSV}")

# Load data
df = pd.read_csv(MASTER_CSV)

print(f"Project Root Detected: {PROJECT_ROOT}")
print(f"Total Master Records: {len(df):,} | Total Attributes: {df.shape[1]}\n")

# 1. Missing Values & Translation Coverage
print("--- Translation Coverage ---")
nulls = df.isnull().sum()
coverage_df = pd.DataFrame(
    {
        "Missing": nulls,
        "Missing (%)": ((nulls / len(df)) * 100).round(2),
        "Present": len(df) - nulls,
    }
)
display(coverage_df)

# 2. Language x Granularity Matrix
print("\n--- Distribution (Language x Granularity) ---")
display(
    pd.crosstab(
        df["language_code"],
        df["level"],
        margins=True,
        margins_name="Total",
    )
)

# 3. Word Count Statistics
df["word_count"] = df["text_native"].fillna("").apply(lambda x: len(str(x).split()))
print("\n--- Word Count Metrics by Language ---")
display(
    df.groupby("language_code")["word_count"]
    .agg(["count", "min", "mean", "median", "max"])
    .round(2)
)

Project Root Detected: c:\Users\Seheltech\Projects\mauritanian-langs
Total Master Records: 9,612 | Total Attributes: 10

--- Translation Coverage ---


,Missing,Missing (%),Present
master_id,0,0.00,9612
language_code,0,0.00,9612
text_native,0,0.00,9612
text_arabic,1535,15.97,8077
text_french,4313,44.87,5299
text_english,7616,79.23,1996
category,0,0.00,9612
level,0,0.00,9612
source_name,0,0.00,9612
source_url,4042,42.05,5570



--- Distribution (Language x Granularity) ---


level,paragraph,phrase,sentence,word,Total
language_code,,,,,
fuc,81,59,2980,0,3120
mey,0,2089,100,0,2189
snk,10,0,749,1553,2312
wol,0,0,1991,0,1991
Total,91,2148,5820,1553,9612



--- Word Count Metrics by Language ---


,count,min,mean,median,max
language_code,,,,,
fuc,3120,1,33.32,25.0,454
mey,2189,1,4.93,4.0,21
snk,2312,1,8.53,1.0,249
wol,1991,1,52.72,34.0,255


In [3]:
# Cell 3: Quality Diagnostics & Quarantine Deep-Dive

QUARANTINE_CSV = (
    PROJECT_ROOT / "data" / "processed" / "reports" / "quarantine.csv"
)

if QUALITY_CSV.exists():
    df_quality = pd.read_csv(QUALITY_CSV)
    print("=== TOP DATA QUALITY FLAGS TRIGGERED ===")
    all_flags = df_quality["flags"].dropna().str.split(";").explode()
    print(all_flags.value_counts())

    print("\n=== SAMPLE FLAGGED RECORDS FOR REVIEW ===")
    display(df_quality[df_quality["flags"] != ""].head(5))

if QUARANTINE_CSV.exists():
    df_quar = pd.read_csv(QUARANTINE_CSV)
    print(
        f"\n=== QUARANTINED ROWS ({len(df_quar)} rows isolated from master) ==="
    )
    display(df_quar)

=== TOP DATA QUALITY FLAGS TRIGGERED ===
flags
long_text                       1763
repetition_loop                  221
footnote_marks                   160
pulaar_typing_substitute          57
native_equals_arabic              15
arabic_column_contains_latin      15
arabic_column_has_no_arabic        8
native_script_mismatch             1
Name: count, dtype: int64

=== SAMPLE FLAGGED RECORDS FOR REVIEW ===


,master_id,completeness_score,flags,words_native,words_arabic,category_is_default,n_source_rows,merged_from
0,HASS-0001,0.8,NaN,2,3,True,1,arabic_hassaniya.csv:2
1,HASS-0002,0.8,NaN,1,1,True,1,arabic_hassaniya.csv:3
2,HASS-0003,0.8,NaN,3,5,True,1,arabic_hassaniya.csv:4
3,HASS-0004,0.8,NaN,5,4,True,1,arabic_hassaniya.csv:5
4,HASS-0005,0.8,NaN,4,5,True,1,arabic_hassaniya.csv:6



=== QUARANTINED ROWS (17 rows isolated from master) ===


,master_id,language_code,text_native,text_arabic,source_name,source_row,reason
0,PUL-1226,fuc,(1),(1),arabic_pulaar.csv,1227,no_letters_in_native_text
1,PUL-1228,fuc,(2),(2),arabic_pulaar.csv,1229,no_letters_in_native_text
2,PUL-1231,fuc,(3),(3),arabic_pulaar.csv,1232,no_letters_in_native_text
3,PUL-1234,fuc,(4),(4),arabic_pulaar.csv,1235,no_letters_in_native_text
4,PUL-1241,fuc,(5),(5),arabic_pulaar.csv,1242,no_letters_in_native_text
5,PUL-1260,fuc,...,..,arabic_pulaar.csv,1261,no_letters_in_native_text
6,PUL-1536,fuc,6130572422001,6130572422001,arabic_pulaar.csv,1537,no_letters_in_native_text
7,PUL-1669,fuc,NaN,NaN,arabic_pulaar.csv,1670,empty_native_text
8,PUL-1674,fuc,NaN,NaN,arabic_pulaar.csv,1675,empty_native_text
9,PUL-1732,fuc,(6),(6),arabic_pulaar.csv,1733,no_letters_in_native_text


In [4]:
# Cell 4: Conflict Resolution & Fuzzy Near-Duplicates

CONFLICTS_CSV = (
    PROJECT_ROOT / "data" / "processed" / "reports" / "merge_conflicts.csv"
)
NEAR_DUPS_CSV = (
    PROJECT_ROOT / "data" / "processed" / "reports" / "near_duplicates.csv"
)

if CONFLICTS_CSV.exists():
    df_conflicts = pd.read_csv(CONFLICTS_CSV)
    print(
        f"=== MULTI-SOURCE CONFLICTS LOGGED ({len(df_conflicts)} instances) ==="
    )
    display(df_conflicts.head(10))

if NEAR_DUPS_CSV.exists():
    df_near = pd.read_csv(NEAR_DUPS_CSV)
    print(
        f"\n=== NEAR-DUPLICATE PAIRS FOR HUMAN REVIEW ({len(df_near)} pairs) ==="
    )
    display(df_near.head(10))

=== MULTI-SOURCE CONFLICTS LOGGED (386 instances) ===


,language_code,text_native,master_id_kept,field,value_kept,values_dropped,files
0,mey,أمنين هي الكنيسة؟,HASS-0075,text_arabic,أين الكاتدرائية ؟,أين الدير ؟,arabic_hassaniya.csv
1,mey,أمنين هي الكنيسة؟,HASS-0075,text_french,Où est la cathédrale?,Où est le couvent?,arabic_hassaniya.csv
2,mey,أمنين هي الكنيسة؟,HASS-0075,text_english,Where is the cathedral?,Where is the monastery?,arabic_hassaniya.csv
3,mey,طحت من المطاليع.,HASS-0353,text_arabic,لقد سقطت من على درجات السلم .,سقطت من على السلم .,arabic_hassaniya.csv
4,mey,طحت من المطاليع.,HASS-0353,text_english,I fell down the stairs.,I fell from the stairs.,arabic_hassaniya.csv
5,mey,ماني متمونك.,HASS-0934,text_arabic,أشعر بوعكة .,أشعر بالإعياء .,arabic_hassaniya.csv
6,fuc,Huutoraade otooji jamaa,PUL-0987,text_arabic,استعمال وسائل النقل الجماعي,استخدام وسائل النقل الجماعي,arabic_pulaar.csv
7,fuc,Huutoraade otooji jamaa,PUL-0987,source_url,https://mawdoo3.com/%D9%83%D9%8A%D9%...,https://mawdoo3.com/%D9%83%D9%8A%D9%...,arabic_pulaar.csv
8,fuc,A mbali jam,PUL-1418,text_arabic,صباح الخير,صَبَاح الْخَيْر,arabic_pulaar.csv; arabic_hassaniya_...
9,fuc,A mbali jam,PUL-1418,level,sentence,phrase,arabic_pulaar.csv; arabic_hassaniya_...



=== NEAR-DUPLICATE PAIRS FOR HUMAN REVIEW (103 pairs) ===


,language_code,master_id_a,text_a,master_id_b,text_b,similarity,reason
0,fuc,PUL-1175,Wawru,PUL-1979,Wawru:,1.000,same_after_folding
1,fuc,PARA-PUL-0824,Com !,PARA-PUL-0825,Com .,1.000,same_after_folding
2,fuc,PARA-PUL-0830,Conce he daaɗe jooɗiiɓe yimooɓe njil...,PARA-PUL-0854,"Coñce he daaɗe jooɗiiɓe yimooɓe , nj...",1.000,same_after_folding
3,fuc,PARA-PUL-0841,Cosgol cate e nder gure ina hollita ...,PARA-PUL-0842,Cosgol cate e nder gure ina hollita ...,1.000,same_after_folding
4,fuc,PARA-PUL-0325,Biritannaaɓe karmininjulaagu ɓaleeɓe .,PARA-PUL-0324,Biritannaaɓe karmini njulaagu ɓaleeɓe .,0.986,fuzzy_match
5,fuc,PARA-PUL-0856,Coñce he dille fitiram golle fiytiro...,PARA-PUL-0855,Coñce he dille fitiram golle en fiyt...,0.976,fuzzy_match
6,fuc,PARA-PUL-0852,Coñce haa ɗo baalnooɗo he naale fina...,PARA-PUL-0853,Coñce haa ɗo baalnooɗo he naale fini...,0.972,fuzzy_match
7,fuc,PARA-PUL-0429,"Caggal jonte ɗiɗi deftagol , sukaabe...",PARA-PUL-0430,"Caggal jonte ɗiɗi deftagol , sukaabe...",0.963,fuzzy_match
8,fuc,PARA-PUL-0851,"Coñce haa ñalla waala , fina kadi da...",PARA-PUL-0850,"Coñce haa haa ñalla waala , fina kad...",0.949,fuzzy_match
9,fuc,PUL-1572,Juulirde nde woni ɗiɗaɓere ɓurnde ma...,PUL-1570,Juulirde Crystal ko juulirde ɗiɗaɓer...,0.932,fuzzy_match


In [5]:
# Self-healing path resolution to locate project root
from pathlib import Path
import pandas as pd

# Configure Pandas for full text inspection
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed" / "master_cleaned.csv").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

MASTER_CSV = PROJECT_ROOT / "data" / "processed" / "master_cleaned.csv"
QUARANTINE_CSV = (
    PROJECT_ROOT / "data" / "processed" / "reports" / "quarantine.csv"
)
CONFLICTS_CSV = (
    PROJECT_ROOT / "data" / "processed" / "reports" / "merge_conflicts.csv"
)

# 1. LOAD & DISPLAY GLOBAL MASTER TABLE SUMMARY
print("=" * 80)
print("1. GLOBAL MASTER TABLE (master_cleaned.csv)")
print("=" * 80)
if MASTER_CSV.exists():
    df_master = pd.read_csv(MASTER_CSV)
    print(
        f"Total Clean Records Staged: {len(df_master):,} rows | {df_master.shape[1]} columns\n"
    )
    print("Sample Master Rows (First 5):")
    display(
        df_master[
            [
                "master_id",
                "language_code",
                "level",
                "text_native",
                "text_arabic",
                "text_french",
            ]
        ].head(5)
    )
else:
    print(f"Error: Master file not found at {MASTER_CSV}")

# 2. LOAD & DISPLAY ELIMINATED / QUARANTINED ROWS WITH REASONS
print("\n" + "=" * 80)
print("2. ELIMINATED / QUARANTINED ROWS AUDIT (reports/quarantine.csv)")
print("=" * 80)
if QUARANTINE_CSV.exists():
    df_quarantine = pd.read_csv(QUARANTINE_CSV)
    print(f"Total Eliminated Records: {len(df_quarantine)} rows\n")

    print("Summary of Elimination Reasons:")
    print(df_quarantine["reason"].value_counts().to_string())

    print("\nComplete List of Eliminated Rows & Their Specific Errors:")
    display(
        df_quarantine[
            [
                "master_id",
                "language_code",
                "source_name",
                "source_row",
                "reason",
                "text_native",
                "text_arabic",
            ]
        ]
    )
else:
    print(f"Error: Quarantine report not found at {QUARANTINE_CSV}")

# 3. LOAD & DISPLAY DROPPED CONFLICTING VALUES DURING MERGING
print("\n" + "=" * 80)
print("3. DROPPED DUP VALUES AUDIT (reports/merge_conflicts.csv)")
print("=" * 80)
if CONFLICTS_CSV.exists():
    df_conflicts = pd.read_csv(CONFLICTS_CSV)
    print(
        f"Total Value Conflicts Overridden During Deduplication: {len(df_conflicts)} instances\n"
    )
    if len(df_conflicts) > 0:
        print("Sample Merged Overrides (First 10):")
        display(
            df_conflicts[
                [
                    "language_code",
                    "master_id_kept",
                    "field",
                    "value_kept",
                    "values_dropped",
                    "files",
                ]
            ].head(10)
        )
else:
    print(f"Error: Merge conflicts report not found at {CONFLICTS_CSV}")

1. GLOBAL MASTER TABLE (master_cleaned.csv)
Total Clean Records Staged: 9,612 rows | 10 columns

Sample Master Rows (First 5):


,master_id,language_code,level,text_native,text_arabic,text_french
0,HASS-0001,mey,phrase,أنگد نگبظها.,يمكنني أن آخذها.,Je peux le prendre.
1,HASS-0002,mey,phrase,الصحة.,لصحتك.,A votre santé.
2,HASS-0003,mey,phrase,حاس بوجيع اكرشي,هناك اضطراب في معدتي .,J'ai l'estomac dérangé.
3,HASS-0004,mey,phrase,أندور أبلاصة أعلا الزر العسري.,أريد مقعداً على اليسار.,Je veux une place sur la gauche.
4,HASS-0005,mey,phrase,نبقي نگبظ مسك لوسيوه.,أوصي بالعطر أو اللوسيون .,Je recommanderais du parfum ou de la lotion.



2. ELIMINATED / QUARANTINED ROWS AUDIT (reports/quarantine.csv)
Total Eliminated Records: 17 rows

Summary of Elimination Reasons:
reason
no_letters_in_native_text    15
empty_native_text             2

Complete List of Eliminated Rows & Their Specific Errors:


,master_id,language_code,source_name,source_row,reason,text_native,text_arabic
0,PUL-1226,fuc,arabic_pulaar.csv,1227,no_letters_in_native_text,(1),(1)
1,PUL-1228,fuc,arabic_pulaar.csv,1229,no_letters_in_native_text,(2),(2)
2,PUL-1231,fuc,arabic_pulaar.csv,1232,no_letters_in_native_text,(3),(3)
3,PUL-1234,fuc,arabic_pulaar.csv,1235,no_letters_in_native_text,(4),(4)
4,PUL-1241,fuc,arabic_pulaar.csv,1242,no_letters_in_native_text,(5),(5)
5,PUL-1260,fuc,arabic_pulaar.csv,1261,no_letters_in_native_text,...,..
6,PUL-1536,fuc,arabic_pulaar.csv,1537,no_letters_in_native_text,6130572422001,6130572422001
7,PUL-1669,fuc,arabic_pulaar.csv,1670,empty_native_text,NaN,NaN
8,PUL-1674,fuc,arabic_pulaar.csv,1675,empty_native_text,NaN,NaN
9,PUL-1732,fuc,arabic_pulaar.csv,1733,no_letters_in_native_text,(6),(6)



3. DROPPED DUP VALUES AUDIT (reports/merge_conflicts.csv)
Total Value Conflicts Overridden During Deduplication: 386 instances

Sample Merged Overrides (First 10):


,language_code,master_id_kept,field,value_kept,values_dropped,files
0,mey,HASS-0075,text_arabic,أين الكاتدرائية ؟,أين الدير ؟,arabic_hassaniya.csv
1,mey,HASS-0075,text_french,Où est la cathédrale?,Où est le couvent?,arabic_hassaniya.csv
2,mey,HASS-0075,text_english,Where is the cathedral?,Where is the monastery?,arabic_hassaniya.csv
3,mey,HASS-0353,text_arabic,لقد سقطت من على درجات السلم .,سقطت من على السلم .,arabic_hassaniya.csv
4,mey,HASS-0353,text_english,I fell down the stairs.,I fell from the stairs.,arabic_hassaniya.csv
5,mey,HASS-0934,text_arabic,أشعر بوعكة .,أشعر بالإعياء .,arabic_hassaniya.csv
6,fuc,PUL-0987,text_arabic,استعمال وسائل النقل الجماعي,استخدام وسائل النقل الجماعي,arabic_pulaar.csv
7,fuc,PUL-0987,source_url,https://mawdoo3.com/%D9%83%D9%8A%D9%81_%D9%86%D8%AD%D8%A7%D9%81%D8%B8_%D8%B9%D9%84%D9%89_%D9%85%D9%88%D8%A7%D8%B1%D8%AF_%D8%A7%D9%84%D8%A8%D9%8A%D8%A6%D8%A9,https://mawdoo3.com/%D9%83%D9%8A%D9%81_%D9%86%D8%AD%D8%A7%D9%81%D8%B8_%D8%B9%D9%84%D9%89_%D8%A7%D9%84%D9%85%D9%88%D8%A7%D8%B1%D8%AF_%D8%A7%D9%84%D8%B7%D8%A8%D9%8A%D8%B9%D9%8A%D8%A9,arabic_pulaar.csv
8,fuc,PUL-1418,text_arabic,صباح الخير,صَبَاح الْخَيْر,arabic_pulaar.csv; arabic_hassaniya_pulaar_trilingual.csv
9,fuc,PUL-1418,level,sentence,phrase,arabic_pulaar.csv; arabic_hassaniya_pulaar_trilingual.csv


In [6]:
from pathlib import Path
import pandas as pd

# Set formatting for full raw content visibility
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

# Self-healing path resolution
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (
        PROJECT_ROOT / "data" / "processed" / "reports" / "quarantine.csv"
    ).exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
QUARANTINE_CSV = (
    PROJECT_ROOT / "data" / "processed" / "reports" / "quarantine.csv"
)

if not QUARANTINE_CSV.exists():
    print(f"Error: Quarantine file not found at {QUARANTINE_CSV}")
else:
    df_quar = pd.read_csv(QUARANTINE_CSV)

    # Map raw filenames to their paths inside data/raw/
    raw_file_map = {p.name.lower(): p for p in RAW_DIR.rglob("*.csv")}

    print(
        f"=== EXTRACTING FULL ORIGINAL RAW ROWS FOR {len(df_quar)} QUARANTINED ENTRIES ===\n"
    )

    for idx, row in df_quar.iterrows():
        file_name = str(row["source_name"]).strip().lower()
        excel_row = int(row["source_row"])  # 1-indexed (header is row 1)
        reason = row["reason"]
        master_id = row["master_id"]

        print(
            f"[{idx+1}/{len(df_quar)}] ID: {master_id} | Reason: {reason} | File: {row['source_name']} (Row {excel_row})"
        )

        if file_name in raw_file_map:
            raw_path = raw_file_map[file_name]
            try:
                # Read raw file as text strings
                df_raw = pd.read_csv(
                    raw_path,
                    dtype=str,
                    keep_default_na=False,
                    encoding="utf-8-sig",
                )

                # Excel row 2 is index 0 in pandas
                df_index = excel_row - 2

                if 0 <= df_index < len(df_raw):
                    raw_row_data = df_raw.iloc[df_index].to_dict()
                    print("   RAW CSV ROW CONTENT:")
                    for col_name, val in raw_row_data.items():
                        print(f"     ├── {col_name}: {repr(val)}")
                else:
                    print(
                        f"   [!] Row index {df_index} out of bounds for {file_name}"
                    )
            except Exception as e:
                print(f"   [!] Error reading file: {e}")
        else:
            print(f"   [!] Could not locate raw file: {file_name}")

        print("-" * 85)

=== EXTRACTING FULL ORIGINAL RAW ROWS FOR 17 QUARANTINED ENTRIES ===

[1/17] ID: PUL-1226 | Reason: no_letters_in_native_text | File: arabic_pulaar.csv (Row 1227)
   RAW CSV ROW CONTENT:
     ├── id: '1226'
     ├── arabic: '(1)'
     ├── pulaar: '(1)'
     ├── category: 'History'
     ├── source_url: 'https://learning.aljazeera.net/en/languageofmedia/%D9%81%D8%A4%D8%A7%D8%AF-%D8%B3%D9%8A%D8%B2%D9%83%D9%8A%D9%86-%D9%85%D9%88%D8%B3%D9%88%D8%B9%D8%A9-%D8%A7%D9%84%D8%AA%D8%B1%D8%A7%D8%AB-%D8%A7%D9%84%D8%B9%D8%B1%D8%A8%D9%8A-%D9%88%D8%A5%D9%86%D8%AC%D8%A7%D8%B2%D8%A7%D8%AA-%D8%A7%D9%84%D9%85%D8%B3%D9%84%D9%85%D9%8A%D9%86'
-------------------------------------------------------------------------------------
[2/17] ID: PUL-1228 | Reason: no_letters_in_native_text | File: arabic_pulaar.csv (Row 1229)
   RAW CSV ROW CONTENT:
     ├── id: '1228'
     ├── arabic: '(2)'
     ├── pulaar: '(2)'
     ├── category: 'History'
     ├── source_url: 'https://learning.aljazeera.net/en/languageofmedia/%D9%

In [2]:
import pandas as pd
chemin=r"C:\Users\Seheltech\Projects\mauritanian-langs\data\processed\master_cleaned.csv"
df=pd.read_csv(chemin)

In [4]:
df

,master_id,language_code,text_native,text_arabic,text_french,text_english,category,level,source_name,source_url
0,HASS-0001,mey,أنگد نگبظها.,يمكنني أن آخذها.,Je peux le prendre.,I can take it.,Conversational,phrase,arabic_hassaniya.csv,NaN
1,HASS-0002,mey,الصحة.,لصحتك.,A votre santé.,To your health.,Conversational,phrase,arabic_hassaniya.csv,NaN
2,HASS-0003,mey,حاس بوجيع اكرشي,هناك اضطراب في معدتي .,J'ai l'estomac dérangé.,I have an upset stomach.,Conversational,phrase,arabic_hassaniya.csv,NaN
3,HASS-0004,mey,أندور أبلاصة أعلا الزر العسري.,أريد مقعداً على اليسار.,Je veux une place sur la gauche.,I want a seat on the left.,Conversational,phrase,arabic_hassaniya.csv,NaN
4,HASS-0005,mey,نبقي نگبظ مسك لوسيوه.,أوصي بالعطر أو اللوسيون .,Je recommanderais du parfum ou de la lotion.,I recommend perfume or lotion.,Conversational,phrase,arabic_hassaniya.csv,NaN
...,...,...,...,...,...,...,...,...,...,...
9607,WOL-1996,wol,Jëfandikukat bi dafa bëgg a jébbal ab jàngu bu...,ويسعى المشروع كذلك لتوفير دراسة تجريبية للتاري...,NaN,NaN,General,sentence,arabic_wolof.csv,https://learning.aljazeera.net/en/languageofme...
9608,WOL-1997,wol,"Te téere yi ñu xamul, te ñu woon woon woon woo...",وسيتم تمييز النصوص غير المعروفة وغير المنشورة ...,NaN,NaN,General,sentence,arabic_wolof.csv,https://learning.aljazeera.net/en/languageofme...
9609,WOL-1998,wol,"Xët yi ci làkk, taariik, téereb téereb, làkk y...",وسيتم دمج اختصاصات فقه اللغة والتاريخ ودراسات ...,NaN,NaN,General,sentence,arabic_wolof.csv,https://learning.aljazeera.net/en/languageofme...
9610,WOL-1999,wol,Ñi doon jël am na xëtu ci li mu bëgg a jël lap...,يقع المرء في حيرة من أمره عندما يرغب في شراء ح...,NaN,NaN,General,sentence,arabic_wolof.csv,NaN
